# Sports Injury Risk Prediction Project
**Objective:** Determine whether we can accurately predict an athlete's risk of injury using machine learning models.

## Steps for Usage

1. **Get the dataset:** Download SIRP-600 from Kaggle and upload to Colab using Cell 2
2. **Run Cells 4-5** to prepare data and split for training
3. **Train models** in section 6
4. **Evaluate** in section 7
5. **Visualize results** in sections 9-10
6. **Fine-tune** with GridSearchCV if needed (section 8)


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Data from Kaggle

In [2]:
#from google.colab import files
#uploaded = files.upload()

#df = pd.read_excel('SIRP-600.xlsx')
#print(df.head())

url = "https://raw.githubusercontent.com/CharlizeCo/CSC_398_SIRP_Project/refs/heads/main/High_Accuracy_Sport_Injury_Dataset.csv"
df = pd.read_csv(url)
df.head(10)

,Age,Gender,Height_cm,Weight_kg,BMI,Training_Frequency,Training_Duration,Warmup_Time,Sleep_Hours,Flexibility_Score,Muscle_Asymmetry,Recovery_Time,Injury_History,Stress_Level,Training_Intensity,Injury_Risk
0,36,0,155.4,56.3,23.34,1,94,20,7.2,63.3,3.7,63,1,8,4.4,0
1,30,0,167.6,45.3,16.12,4,114,5,7.1,64.6,5.1,64,1,7,6.7,0
2,21,1,176.7,60.8,19.48,1,95,11,5.0,68.6,6.1,69,0,6,4.3,1
3,37,0,170.2,60.7,20.97,5,73,6,8.1,69.0,4.6,65,2,3,6.4,1
4,30,0,161.5,45.0,17.25,6,93,8,7.8,45.2,8.5,42,3,8,4.2,1
5,18,1,174.3,74.8,24.61,2,80,2,7.7,60.2,5.8,106,0,3,7.1,0
6,40,0,159.8,68.6,26.84,6,81,6,7.5,67.3,3.1,94,0,3,5.3,0
7,23,0,151.8,54.6,23.69,2,120,3,8.5,75.2,3.6,35,0,8,5.8,0
8,28,0,166.0,52.1,18.91,3,83,11,6.5,53.4,3.3,103,0,10,8.1,1
9,32,1,176.2,58.5,18.87,6,88,18,7.4,71.8,0.8,108,0,1,3.4,0


## 3. Data Exploration

In [11]:
# Display basic info about the dataset

print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")
print(f"\nColumn names and types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

['Age', 'Gender', 'Height_cm', 'Weight_kg', 'BMI', 'Training_Frequency', 'Training_Duration', 'Warmup_Time', 'Sleep_Hours', 'Flexibility_Score', 'Muscle_Asymmetry', 'Recovery_Time', 'Injury_History', 'Stress_Level', 'Training_Intensity', 'Injury_Risk']

Dataset shape: (600, 16)

Column names and types:
Age                     int64
Gender                  int64
Height_cm             float64
Weight_kg             float64
BMI                   float64
Training_Frequency      int64
Training_Duration       int64
Warmup_Time             int64
Sleep_Hours           float64
Flexibility_Score     float64
Muscle_Asymmetry      float64
Recovery_Time           int64
Injury_History          int64
Stress_Level            int64
Training_Intensity    float64
Injury_Risk             int64
dtype: object

Missing values:
Age                   0
Gender                0
Height_cm             0
Weight_kg             0
BMI                   0
Training_Frequency    0
Training_Duration     0
Warmup_Time      

In [4]:
df.describe()

,Age,Gender,Height_cm,Weight_kg,BMI,Training_Frequency,Training_Duration,Warmup_Time,Sleep_Hours,Flexibility_Score,Muscle_Asymmetry,Recovery_Time,Injury_History,Stress_Level,Training_Intensity,Injury_Risk
count,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000
mean,29.053333,0.441667,168.091167,62.816000,22.185433,3.495000,83.180000,9.988333,7.242333,60.232000,5.097333,74.598333,0.605000,5.498333,5.510167,0.315000
std,6.406160,0.497000,8.478391,11.608189,3.457063,1.692557,21.441482,6.186705,0.785198,10.098983,2.848435,26.545020,0.858411,2.912754,1.789754,0.464903
min,18.000000,0.000000,150.000000,45.000000,12.660000,1.000000,45.000000,0.000000,5.000000,30.500000,0.000000,30.000000,0.000000,1.000000,1.000000,0.000000
25%,24.000000,0.000000,161.475000,54.000000,19.720000,2.000000,66.000000,5.000000,6.700000,53.400000,3.100000,52.000000,0.000000,3.000000,4.300000,0.000000
50%,29.000000,0.000000,167.300000,61.200000,22.085000,3.000000,82.000000,10.000000,7.300000,60.350000,5.000000,75.000000,0.000000,5.000000,5.500000,0.000000
75%,34.000000,1.000000,174.725000,70.925000,24.390000,5.000000,101.000000,15.000000,7.800000,66.925000,7.000000,98.250000,1.000000,8.000000,6.800000,1.000000
max,40.000000,1.000000,190.000000,95.000000,33.550000,6.000000,120.000000,20.000000,9.500000,94.600000,14.400000,119.000000,3.000000,10.000000,10.000000,1.000000


## 4. Data Preparation

**Separate features (X) and target (y)**

In [9]:
# Identify target column
features = [
    'Age',
    'Gender',
    'Height_cm',
    'Weight_kg',
    'BMI',
    'Training_Frequency',
    'Training_Duration',
    'Warmup_Time',
    'Sleep_Hours',
    'Flexibility_Score',
    'Muscle_Asymmetry',
    'Recovery_Time',
    'Injury_History',
    'Stress_Level',
    'Training_Intensity'
]

target = 'Injury_Risk'

X = df[features]  # Features
y = df[target]  # Target

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:\n{y.value_counts()}")


Features shape: (600, 15)
Target shape: (600,)

Target distribution:
Injury_Risk
0    411
1    189
Name: count, dtype: int64


600

## 5. Train-Test Split (80-20)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=0.2, random_state=42
 )

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (480, 15)
Testing set size: (120, 15)


## 6. Model Training

### 6.1 Random Forest

In [ ]:
# rf_model = RandomForestClassifier(random_state=42)
# rf_model.fit(X_train, y_train)
# rf_pred = rf_model.predict(X_test)
# print("Random Forest trained!")

### 6.2 Gradient Boosting

In [ ]:
# gb_model = GradientBoostingClassifier(random_state=42)
# gb_model.fit(X_train, y_train)
# gb_pred = gb_model.predict(X_test)
# print("Gradient Boosting trained!")

### 6.3 Support Vector Classifier

In [ ]:
# svc_model = SVC(kernel='rbf', random_state=42, probability=True)
# svc_model.fit(X_train, y_train)
# svc_pred = svc_model.predict(X_test)
# print("Support Vector Classifier trained!")

## 7. Model Evaluation

**Metrics:** Accuracy, Precision, Recall, ROC-AUC

In [ ]:
# def evaluate_model(model_name, y_test, y_pred, y_pred_proba=None):
#     accuracy = accuracy_score(y_test, y_pred)
#     precision = precision_score(y_test, y_pred, zero_division=0)
#     recall = recall_score(y_test, y_pred, zero_division=0)
#
#     print(f"\n{model_name} Performance:")
#     print(f"  Accuracy:  {accuracy:.4f}")
#     print(f"  Precision: {precision:.4f}")
#     print(f"  Recall:    {recall:.4f}")
#
#     return accuracy, precision, recall

# # Evaluate all models
# rf_acc, rf_prec, rf_rec = evaluate_model('Random Forest', y_test, rf_pred)
# gb_acc, gb_prec, gb_rec = evaluate_model('Gradient Boosting', y_test, gb_pred)
# svc_acc, svc_prec, svc_rec = evaluate_model('SVC', y_test, svc_pred)

## 8. Hyperparameter Tuning (Optional - GridSearchCV)

**fill this in once we see which model performs best**

In [ ]:
# Example for Random Forest (uncomment when ready):
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [5, 10, 15]
# }
# grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
# grid_search.fit(X_train, y_train)
# best_model = grid_search.best_estimator_
# print(f"Best parameters: {grid_search.best_params_}")

## 9. Visualizations

**Use this to create charts here once models are trained**

In [ ]:
# Example: Compare model accuracy
# models = ['Random Forest', 'Gradient Boosting', 'SVC']
# accuracies = [rf_acc, gb_acc, svc_acc]
#
# plt.figure(figsize=(8, 5))
# plt.bar(models, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
# plt.ylabel('Accuracy')
# plt.title('Model Comparison: Accuracy')
# plt.ylim([0, 1])
# plt.show()

## 10. ROC Curves

In [ ]:
# Example ROC curve for one model
# y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
# fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
# roc_auc = auc(fpr, tpr)
#
# plt.figure(figsize=(8, 6))
# plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
# plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve - Random Forest')
# plt.legend()
# plt.show()